# 05 · Calculate area, checked against spherical and geodesic areas

## What this checks

**Calculate area** measures the area of a boundary. This check covers release 0.3.0.

Fieldwork says how it does this. It uses Turf 7.3.5, a map library for the browser. Turf treats the Earth as a
sphere with a mean radius of 6,371,008.8 m. Fieldwork calls the result approximate, because the Earth is not quite a
sphere.

An area is often the denominator of a rate, so an error in it moves the rate. So it is worth checking.

## How it is checked

This notebook asks three separate questions. Mixing them would report a choice of method as a defect.

1. **Does the widget compute the formula it declares?** We work out the same spherical area with a different formula,
   written here. The boundaries follow lines of latitude and longitude, where that formula is exact.
2. **How much do the edges matter?** We compare with a geodesic area on the same sphere. A geodesic joins the corners
   by the shortest path over the surface, a great circle. This is a difference of model, reported and never failed.
3. **How much does the shape of the Earth matter?** We compare with an area on WGS84, the ellipsoid GPS uses. It is
   slightly flattened at the poles. This measures the approximation Fieldwork already declares.

The geodesic areas come from **pyproj**, using GeographicLib, a standard library for distances and areas on the Earth.
Nothing here uses Fieldwork's code. The data carry the settings Fieldwork declared and its own output. Their digest is
checked when they are loaded.

The data are five boundaries. One of them has a hole.

## What should happen

1. For each boundary, Fieldwork's area and the area from our own formula differ by at most one part in a billion.

Only the first question has a pass or fail. The other two are measured and reported. This criterion was fixed before
the check was run.

In [ ]:
# pyproj ships in the Pyodide distribution, not as a PyPI wheel on this site, so it loads
# with pyodide_js.loadPackage. piplite.install would look in the site's local wheel index
# and fail. Outside Pyodide this cell does nothing.
NEEDED = ["pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_05_area import run

report = run(Path.cwd() / "fixtures" / "measure-area-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(report["fixture"]["declaredMethod"])


## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.3.0:
**Agrees**.

The two spherical areas differ by at most about 5 parts in 10¹⁴. That is far inside one part in a billion. The widget
computes the formula it says it computes.

Running the cell above repeats the check on the same data.

## The three comparisons side by side

The first column should be near the limit of computer arithmetic. It is the same formula, worked out two ways. The
other two columns are sizes, not verdicts.

In [ ]:
# Three columns, three different questions. The first is the only pass/fail one.
print(f"{'case':<24}{'implementation':>16}{'edge model':>14}{'earth model':>14}")
for case in report["cases"]:
    c = case["context"]
    print(f"{case['id']:<24}"
          f"{case['relativeDifference']:>16.2e}"
          f"{c['relativeToGreatCircleEdges']*100:>13.4f}%"
          f"{c['relativeToEllipsoidal']*100:>13.4f}%")


## The same differences in reporting units

A percentage is easy to dismiss. The cell below gives the difference from WGS84 in square kilometres. That is the
number a report would quote.

In [ ]:
# What the differences mean in the units a report would quote.
for case in report["cases"]:
    c = case["context"]
    km2 = case["fieldworkSquareMetres"] / 1e6
    gap = (case["fieldworkSquareMetres"] - c["wgs84EllipsoidalSquareMetres"]) / 1e6
    print(f"{case['id']:<24}{km2:>12.3f} km2   against WGS84: {gap:+.3f} km2")


## Reading the sizes

**The edges matter only when they run at a slant**. Boxes whose sides follow latitude and longitude differ from
great-circle areas by less than 0.01%. A mid-latitude triangle, with long slanting sides, differs by about **0.3%**.

**The shape of the Earth matters everywhere, and not by a fixed amount**. Against WGS84, the spherical area is about
**0.45% larger at the equator** and about **0.56% smaller at 60° N**. The sign changes with latitude. So no single
factor can correct it.

None of this makes the widget wrong. It declares a spherical approximation. What the check adds is the size of that
approximation, measured. A practitioner can then decide whether 0.2 to 0.6% matters. For the denominator of a rate,
usually not. For a boundary dispute, or a payment based on area, it may.

## What this does not show

- That a spherical area is good enough for a given purpose. The check measures the approximation; it does not judge
  it.
- That the edge and Earth differences are failures. They are measured sizes of a declared approximation.